# NLRB union elections — exploratory data analysis

Loads the `analytical_elections` view (RC/RM filings) and profiles the two
modeling targets: the regression target `vote_ratio` and the classification
target `success`.

In [2]:
import duckdb
import pandas as pd

with duckdb.connect("./data/union_elections.duckdb", read_only=True) as con:
    df = con.execute("SELECT * FROM analytical_elections").fetchdf()
print(df.columns)
print(df.shape)
df.head()

Index(['case_number', 'case_type', 'city', 'state', 'date_filed',
       'date_closed', 'reason_closed', 'election_id', 'tally_type',
       'unit_size', 'election_date', 'more_than_one_election',
       'votes_for_union', 'votes_against', 'rpt_id', 'yr_covered',
       'pd_covered_from', 'pd_covered_to', 'f_num', 'desig_num', 'terminate',
       'representational', 'political', 'contributions', 'general_overhead',
       'union_administration', 'employer_name', 'region_assigned',
       'filing_status', 'filing_number_of_eligible_voters',
       'number_of_voters_on_petition_or_charge', 'certified_representative',
       'ballot_type', 'voting_unit_id', 'finance_union_name',
       'finance_aff_abbr', 'form_type', 'ttl_receipts', 'ttl_disbursements',
       'election_union_name', 'union_participant', 'union_participant_type',
       'pred_union_name', 'pred_f_num', 'pred_f_num_score', 'pred_is_union',
       'pred_union_name_score', 'pred_is_union_score'],
      dtype='str')
(22196, 4

,case_number,case_type,city,state,date_filed,date_closed,reason_closed,election_id,tally_type,unit_size,...,ttl_disbursements,election_union_name,union_participant,union_participant_type,pred_union_name,pred_f_num,pred_f_num_score,pred_is_union,pred_union_name_score,pred_is_union_score
0,25-RC-215961,RC,Grandview,IN,2018-03-05,2018-04-09,Certification of Results,17754,Initial,25,...,32523,"United Steel Workers, Local 6129.","United Steel Workers, Local 6129.",Petitioner,"STEELWORKERS, AFL-CIO",56287,0.9974,1,0.9935,0.9678
1,08-RC-218479,RC,Strongsville,OH,2018-04-16,2018-05-21,Certification of Results,7787,Initial,21,...,1525324,INTERNATIONAL ASSOCIATION OF MACHINISTS AND AE...,INTERNATIONAL ASSOCIATION OF MACHINISTS AND AE...,Petitioner,MACHINISTS AFL-CIO,16682,0.9959,1,0.9952,0.9591
2,31-RC-215803,RC,Los Angeles,CA,2018-03-02,2018-03-30,Certific. of Representative,20290,Initial,183,...,160050,"United Electrical, Radio, and Machine Workers ...","United Electrical, Radio, and Machine Workers ...",Petitioner,ELECTRICAL WORKERS UE IND,37321,0.0542,1,0.9970,0.9230
3,15-RC-218692,RC,Jackson,MS,2018-04-19,2018-05-23,Certific. of Representative,22123,Initial,6,...,38571,Amalgamated Transit Union Local 1208,Amalgamated Transit Union Local 1208,Petitioner,TRANSIT UNION AFL-CIO,13420,0.9925,1,0.9956,0.9561
4,19-RC-221590,RC,Ashland,OR,2018-06-07,2018-07-30,Certification of Results,14275,Initial,50,...,54210,International Alliance of Theatrical Stage Emp...,International Alliance of Theatrical Stage Emp...,Petitioner,STAGE AND PICTURE OPERATORS AFL-CIO,545555,0.9872,1,0.9959,0.9573


## Case types, uniqueness

The view is one row per election (one completed RC/RM case each).

In [4]:
print("rows:", len(df))
print("distinct filings:", df["case_number"].nunique())
print("distinct elections:", df["election_id"].nunique())
print()
df["case_type"].value_counts()

rows: 22196
distinct filings: 22196
distinct elections: 22196



case_type
RC    21598
RM      598
Name: count, dtype: int64

## Closing reason and the success label

An election is treated as successful when
`reason_closed == 'Certific. of Representative'`.

In [28]:
df["reason_closed"].value_counts(dropna=False)

reason_closed
Certific. of Representative    15170
Certification of Results        6615
Withdrawal Non-adjusted          339
Dismissal Non-adjusted            32
Withdrawal Adjusted               28
Administrative Closing             4
Amended Certification              4
Compliance w/BO                    2
Unit Clarification                 1
Dismissal Adjusted                 1
Name: count, dtype: int64

In [5]:
df["success"] = (
    df["reason_closed"]
    .map(lambda value: pd.NA if pd.isna(value) else int(value == "Certific. of Representative"))
    .astype("Int64")
)
print(df["success"].value_counts(dropna=False))

print(f'The fraction of rows that are success is {sum(df["success"]) / len(df)}')

success
1    15170
0     7026
Name: count, dtype: Int64
The fraction of rows that are success is 0.6834564786448009


Relatively mild class imbalance since there are still ~7000 cases that did not succeed.

## Vote counts and the regression target

`vote_ratio = votes_for_union / votes_against`.
Elections with zero votes against give an undefined ratio.

In [6]:
votes = df.dropna(
    subset=["votes_for_union", "votes_against"]
).copy()
print("elections with vote counts:", len(votes))
print("zero votes against:", int((votes["votes_against"] == 0).sum()))

votes["vote_ratio"] = (
    votes["votes_for_union"]
    / votes["votes_against"].replace(0, float("nan"))
)
votes["vote_ratio"].describe()

elections with vote counts: 16771
zero votes against: 0


count     16771.0
mean     3.968546
std      9.239619
min      0.015152
25%          0.75
50%      1.666667
75%           4.0
max         387.0
Name: vote_ratio, dtype: Float64

## Missingness

In [7]:
df.isna().mean().sort_values(ascending=False).head(30)

certified_representative            0.538385
filing_number_of_eligible_voters    0.349207
desig_num                           0.281177
general_overhead                    0.252478
political                           0.252478
union_administration                0.252478
representational                    0.252478
votes_against                       0.225176
contributions                       0.168544
terminate                           0.155028
ttl_receipts                        0.135925
pd_covered_from                     0.135880
yr_covered                          0.135880
pd_covered_to                       0.135880
f_num                               0.135880
rpt_id                              0.135880
finance_aff_abbr                    0.135880
finance_union_name                  0.135880
form_type                           0.135880
ttl_disbursements                   0.135880
votes_for_union                     0.023473
pred_is_union                       0.018472
pred_union

## Finance coverage

In [8]:
(
    df.groupby("form_type", dropna=False)
        .agg(
            rows=("case_number", "size"),
            median_disbursements=("ttl_disbursements", "median"),
            median_receipts=("ttl_receipts", "median"),
        )
        .sort_values("rows", ascending=False)
)

,rows,median_disbursements,median_receipts
form_type,,,
LM-2,16592,5030522.5,5141985.0
NaN,3016,<NA>,<NA>
LM-3,1903,46689.0,56914.0
LM-4,685,500.0,872.0


## Success by case type and state

In [9]:
df.groupby("case_type")["success"].agg(["size", "mean"])

,size,mean
case_type,,
RC,21598,0.688721
RM,598,0.493311


In [11]:
bystate = (
  df.groupby("state")["success"]
    .agg(["size", "mean"])
    .sort_values("size", ascending=False)
)

print(bystate.head(15))

print(bystate.sort_values("mean", ascending=True).head(15))

       size      mean
state                
CA     3179  0.707455
NY     2397  0.727159
IL     1515   0.69571
PA     1398  0.658083
NJ     1042   0.65643
WA      999  0.714715
MI      781   0.62612
MA      776  0.734536
OH      732  0.560109
TX      695  0.689209
OR      612  0.668301
MN      555  0.540541
MD      533  0.739212
CT      484  0.702479
FL      482  0.738589
       size      mean
state                
SD       17  0.411765
IA      146  0.465753
ID       84  0.488095
WY       31  0.516129
AR       63  0.539683
MN      555  0.540541
OH      732  0.560109
MS       62  0.564516
IN      400      0.59
KY      252  0.599206
WI      265       0.6
WV      156  0.608974
MO      461  0.611714
CO      299   0.61204
VI       26  0.615385


## Notes

- Vote counts come from the `tally` table (`votes_for_union` /
  `votes_against`). Elections with a contested (>2 option) or only-"No union"
  tally are excluded.
- Elections with zero or missing votes against have an undefined ratio, so
  they are dropped for regression.
- `training.py` loads the same view, collapses it to one row per election, and
  builds the feature/target frames.